# Routing101 — single-embedding retrieval, no fusion, no routing

**Purpose:** isolate whether the *embedding + search* step works on its own, before any RRF, OCR filtering, preset routing, or S1/S2/S3 logic gets layered on top.

Three backends, tested one at a time:

| Backend | Dim | Frame `.npy` source | Text encoder |
|---|---|---|---|
| `siglip2` | 768 | you provide path to frame embeddings | `google/siglip2-base-patch16-384` text tower |
| `clip_vitb32` | 512 | you provide path to frame embeddings | Multilingual-CLIP text tower |
| `viclip` | 768 | you provide path to frame embeddings | ViCLIP-OT (`minhnguyent546/ViCLIP-OT`) text tower |

Each backend builds its own `IndexFlatIP` FAISS index from raw `.npy` frame embeddings, encodes your query text with that backend's own text tower, and returns a plain ranked list (`video_id`, `frame_id` only — no timestamps). **No cross-backend fusion happens in this notebook** — that's Round 2, later, once each backend is validated alone.

**The only things you need to edit are in Cell 2: the `QUERY` text and the 3 `frame_glob` paths.** Everything else runs as-is, assuming your `.npy` files are named `{video_id}.npy` (or `{video_id}_viclip768.npy`, etc — same stem-stripping as before) and each `.npy` row order matches keyframe order within that video (row 0 = first keyframe, row 1 = second, ...). Frame IDs are just the 0-indexed row position within each video's `.npy` — no CSV needed.

## 1. Install / imports
Run once. Uses `pip install --break-system-packages` if you're on an externally-managed Python env — drop that flag if you're in a normal venv/conda env.

In [17]:
# !pip install --break-system-packages faiss-cpu numpy pandas torch transformers sentencepiece pillow

import os
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import torch

print("faiss:", faiss.__version__ if hasattr(faiss, '__version__') else 'unknown')
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

faiss: 1.15.0
torch: 2.13.0+cpu | cuda available: False


## 2. CONFIG — edit this cell

Set `QUERY` and the 3 `frame_glob` paths. That's it.

Each `frame_glob` should match one `.npy` file per video (e.g. `L21_V001.npy`), where each row is one keyframe's embedding vector, in keyframe order. `video_id` = filename stem, `frame_id` = 0-indexed row number within that file.

In [ ]:
# ============================================================
# EDIT ME
# ============================================================

# QUERY = "Đoạn clip cần tìm là cảnh hai người phụ nữ đang cho dê ăn: một người mặc áo thun trắng quàng áo đỏ trên vai, người kia mặc áo dài tay kẻ sọc tím truyền thống. Cả hai đều mỉm cười, tỏ vẻ thích thú."  # <-- edit me

CONFIG = {
    "siglip2": {
        "enabled": True,
        "frame_glob": "D:/University/Summ26/AICDataExtracted/siglib_embed/*.npy",
        "hf_model_id": "google/siglip2-base-patch16-384",
        "dim": 768,
    },
    "clip_vitb32": {
        "enabled": True,
        "frame_glob": "D:/University/Summ26/AICData/clip-features-32/*.npy",
        "hf_model_id": "M-CLIP/XLM-Roberta-Large-Vit-B-32",  # Multilingual-CLIP text tower
        "dim": 512,
    },
    "viclip": {
        "enabled": True,
        "frame_glob": "D:/University/Summ26/AICDataExtracted/embeddings/*.npy",
        "hf_model_id": "minhnguyent546/ViCLIP-OT",
        "dim": 768,
    },
}

# ============================================================
# below this line: no need to edit for normal use
# ============================================================

def video_id_from_filename(npy_path: str) -> str:
    """Derive video_id from a frame-embedding .npy filename. Edit if your
    naming convention differs (e.g. strip a '_viclip768' suffix, etc)."""
    stem = Path(npy_path).stem
    for suffix in ("_viclip768", "_clip32", "_siglip768", "_siglip2"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
    return stem

TOP_K = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 3. Shared index-building helpers

Builds one `IndexFlatIP` per backend directly from the `.npy` files, plus a parallel lookup table (`global_id -> video_id, frame_id`) — `frame_id` is just the row's position within its video's `.npy` file. No CSV, no timestamp.

In [19]:
def l2_normalize(mat: np.ndarray) -> np.ndarray:
    """IndexFlatIP == cosine similarity only if vectors are L2-normalized first."""
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1e-8
    return mat / norms


def build_backend_index(backend_name: str, cfg: dict):
    """Load all per-video .npy frame embeddings for one backend, concatenate,
    build a FAISS IndexFlatIP, and return (index, lookup_df) where lookup_df
    row i corresponds to index vector i, giving just video_id + frame_id.
    """
    npy_paths = sorted(glob.glob(cfg["frame_glob"]))
    if not npy_paths:
        raise FileNotFoundError(
            f"[{backend_name}] no .npy files matched: {cfg['frame_glob']}\n"
            f"-> check the path in the CONFIG cell."
        )

    all_vecs = []
    lookup_rows = []

    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path)
        vecs = np.load(npy_path).astype("float32")
        if vecs.ndim == 1:
            vecs = vecs.reshape(1, -1)

        for row_idx in range(len(vecs)):
            lookup_rows.append({"video_id": video_id, "frame_id": row_idx})
        all_vecs.append(vecs)

    matrix = np.vstack(all_vecs).astype("float32")
    matrix = l2_normalize(matrix)
    dim = matrix.shape[1]

    if dim != cfg["dim"]:
        print(
            f"[{backend_name}] WARNING: loaded dim={dim} but CONFIG says dim={cfg['dim']}. "
            f"Using the actual loaded dim; double check you pointed at the right files."
        )

    index = faiss.IndexFlatIP(dim)
    index.add(matrix)

    lookup_df = pd.DataFrame(lookup_rows)

    print(f"[{backend_name}] indexed {index.ntotal} frames from {len(npy_paths)} videos, dim={dim}")

    return index, lookup_df


def search_index(index: faiss.Index, lookup_df: pd.DataFrame, query_vec: np.ndarray, k: int = TOP_K) -> pd.DataFrame:
    """Search a built index with a single (already-normalized) query vector.
    Returns a results DataFrame: rank, score, video_id, frame_id.
    """
    q = query_vec.astype("float32").reshape(1, -1)
    q = l2_normalize(q)
    scores, ids = index.search(q, k)
    scores, ids = scores[0], ids[0]

    results = lookup_df.iloc[ids].copy().reset_index(drop=True)
    results["score"] = scores
    results["rank"] = np.arange(1, len(results) + 1)
    return results[["rank", "score", "video_id", "frame_id"]]

## 4. Backend: SigLIP2

Loads `google/siglip2-base-patch16-384`'s text tower for query encoding, and builds the FAISS index from your configured `.npy` files.

In [20]:
siglip2_index = None
siglip2_lookup = None
siglip2_model = None
siglip2_processor = None

if CONFIG["siglip2"]["enabled"]:
    from transformers import AutoModel, AutoProcessor

    siglip2_index, siglip2_lookup = build_backend_index("siglip2", CONFIG["siglip2"])

    siglip2_model = AutoModel.from_pretrained(CONFIG["siglip2"]["hf_model_id"]).to(DEVICE).eval()
    siglip2_processor = AutoProcessor.from_pretrained(CONFIG["siglip2"]["hf_model_id"])


def encode_query_siglip2(text: str) -> np.ndarray:
    inputs = siglip2_processor(text=[text], padding="max_length", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = siglip2_model.get_text_features(**inputs)
    # SiglipModel.get_text_features can return the raw BaseModelOutputWithPooling
    # (delegates straight to the text tower) instead of a plain tensor, depending
    # on the transformers version -- pooler_output is the pooled embedding either way.
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.cpu().numpy()[0]


def search_siglip2(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = encode_query_siglip2(query)
    return search_index(siglip2_index, siglip2_lookup, qvec, k)

[siglip2] indexed 177321 frames from 873 videos, dim=768


Loading weights: 100%|██████████| 408/408 [00:00<00:00, 2536.23it/s]
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/google/siglip2-base-patch16-384/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


## 5. Backend: CLIP ViT-B/32 (Multilingual-CLIP)

In [21]:
clip_index = None
clip_lookup = None
clip_text_model = None
clip_tokenizer = None

if CONFIG["clip_vitb32"]["enabled"]:
    # Multilingual-CLIP ships its own small wrapper package; fall back to a
    # plain transformers AutoModel load if `multilingual_clip` isn't installed.
    clip_index, clip_lookup = build_backend_index("clip_vitb32", CONFIG["clip_vitb32"])

    try:
        from multilingual_clip.pt_multilingual_clip import MultilingualCLIP
        from multilingual_clip.Config_MCLIP import MCLIPConfig
        from transformers import AutoTokenizer
        from huggingface_hub import hf_hub_download

        # MultilingualCLIP.__init__ calls transformers.AutoModel.from_pretrained(...)
        # internally (a nested from_pretrained call). Current transformers always
        # builds the outer model on a meta device first (fast-init), and that
        # nested call errors out ("from_pretrained with a meta device context
        # manager") no matter what kwargs are passed to the outer from_pretrained.
        # Workaround: build the model directly from its config (no ambient meta
        # context, so the nested AutoModel.from_pretrained inside __init__ works
        # normally) and load the checkpoint's weights onto it by hand.
        model_id = CONFIG["clip_vitb32"]["hf_model_id"]
        mclip_config = MCLIPConfig.from_pretrained(model_id)
        clip_text_model = MultilingualCLIP(mclip_config)
        try:
            weights_path = hf_hub_download(model_id, "pytorch_model.bin")
            state_dict = torch.load(weights_path, map_location="cpu")
        except Exception:
            from safetensors.torch import load_file
            weights_path = hf_hub_download(model_id, "model.safetensors")
            state_dict = load_file(weights_path)
        clip_text_model.load_state_dict(state_dict, strict=False)
        clip_text_model = clip_text_model.to(DEVICE).eval()

        clip_tokenizer = AutoTokenizer.from_pretrained(model_id)
        _clip_backend_kind = "multilingual_clip_pkg"
    except ImportError:
        print(
            "multilingual_clip package not found — install with:\n"
            "  pip install --break-system-packages multilingual-clip\n"
            "Falling back to a generic transformers load, which may not match "
            "the exact text tower used to build your .npy files if they came "
            "from the multilingual_clip package originally."
        )
        from transformers import AutoModel, AutoTokenizer

        clip_text_model = AutoModel.from_pretrained(CONFIG["clip_vitb32"]["hf_model_id"]).to(DEVICE).eval()
        clip_tokenizer = AutoTokenizer.from_pretrained(CONFIG["clip_vitb32"]["hf_model_id"])
        _clip_backend_kind = "transformers_generic"


def encode_query_clip(text: str) -> np.ndarray:
    if _clip_backend_kind == "multilingual_clip_pkg":
        with torch.no_grad():
            feats = clip_text_model.forward([text], clip_tokenizer)
        return feats.cpu().numpy()[0]
    else:
        inputs = clip_tokenizer([text], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        with torch.no_grad():
            out = clip_text_model(**inputs)
        feats = out.pooler_output if hasattr(out, "pooler_output") else out.last_hidden_state[:, 0, :]
        return feats.cpu().numpy()[0]


def search_clip(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = encode_query_clip(query)
    return search_index(clip_index, clip_lookup, qvec, k)

[clip_vitb32] indexed 177321 frames from 873 videos, dim=512


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4718.15it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `M-CLIP` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


## 6. Backend: ViCLIP-OT

In [22]:
viclip_index = None
viclip_lookup = None
viclip_model = None
viclip_tokenizer = None

if CONFIG["viclip"]["enabled"]:
    from transformers import AutoModel, AutoTokenizer

    viclip_index, viclip_lookup = build_backend_index("viclip", CONFIG["viclip"])

    viclip_model = AutoModel.from_pretrained(
        CONFIG["viclip"]["hf_model_id"], trust_remote_code=True
    ).to(DEVICE).eval()
    viclip_tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["viclip"]["hf_model_id"], trust_remote_code=True
    )


def encode_query_viclip(text: str) -> np.ndarray:
    inputs = viclip_tokenizer([text], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        # ViCLIP-OT exposes a text tower — adjust the call below if the actual
        # HF repo's API differs (check model card / config.json on first run,
        # the exact method name can vary by checkpoint).
        if hasattr(viclip_model, "get_text_features"):
            feats = viclip_model.get_text_features(**inputs)
        else:
            out = viclip_model(**inputs)
            feats = out.text_embeds if hasattr(out, "text_embeds") else out.last_hidden_state[:, 0, :]
    return feats.cpu().numpy()[0]


def search_viclip(query: str, k: int = TOP_K) -> pd.DataFrame:
    qvec = encode_query_viclip(query)
    return search_index(viclip_index, viclip_lookup, qvec, k)

[viclip] indexed 177321 frames from 873 videos, dim=768


Loading weights: 100%|██████████| 363/363 [00:00<00:00, 2396.06it/s]


## 7. Run a single query — Checkpoint A/B side by side

Uses the `QUERY` you set in Cell 2. To try a new query, edit `QUERY` up in Cell 2 and re-run from there (or just reassign `QUERY` below and re-run this cell). Each backend runs completely independently — no fusion happens here.

In [ ]:
QUERY = "cô bé đeo một con bạch tuộc / con mực màu đỏ phía trước ngực"  # uncomment to override without touching Cell 2
SHOW_TOP_N = 15

results_by_backend = {}

if siglip2_index is not None:
    results_by_backend["siglip2"] = search_siglip2(QUERY, k=TOP_K)

if clip_index is not None:
    results_by_backend["clip_vitb32"] = search_clip(QUERY, k=TOP_K)

for name, df in results_by_backend.items():
    print(f"\n=== {name} — top {SHOW_TOP_N} ===")
    display(df.head(SHOW_TOP_N))


=== siglip2 — top 15 ===


,rank,score,video_id,frame_id
0,1,0.197186,L22_V030_siglip768,193
1,2,0.179398,L30_V022_siglip768,16
2,3,0.167959,L22_V025_siglip768,122
3,4,0.151997,L22_V012_siglip768,254
4,5,0.149529,L22_V001_siglip768,157
5,6,0.148163,L24_V019_siglip768,69
6,7,0.146478,L24_V037_siglip768,140
7,8,0.146450,L22_V001_siglip768,163
8,9,0.146121,L30_V022_siglip768,15
9,10,0.146026,L24_V023_siglip768,38



=== clip_vitb32 — top 15 ===


,rank,score,video_id,frame_id
0,1,0.313350,L22_V001,157
1,2,0.302001,L22_V014,77
2,3,0.300255,L30_V047,69
3,4,0.299413,L22_V008,25
4,5,0.299055,L30_V047,82
5,6,0.297813,L30_V047,70
6,7,0.295828,L30_V015,21
7,8,0.294914,L21_V013,218
8,9,0.294439,L22_V010,112
9,10,0.292235,L22_V012,254



=== viclip — top 15 ===


,rank,score,video_id,frame_id
0,1,0.652970,L24_V038,81
1,2,0.652495,L30_V047,82
2,3,0.652438,L26_V338,159
3,4,0.646700,L24_V038,80
4,5,0.643297,L27_V010,291
5,6,0.643036,L26_V467,81
6,7,0.642816,L26_V384,27
7,8,0.642297,L29_V002,275
8,9,0.641922,L28_V023,290
9,10,0.640486,L26_V499,17


## 8. Score-distribution sanity check

Quick gut-check per backend: are top-K scores well separated from the tail, or basically flat (a sign of a near-random / unhelpful embedding space for this query)? This mirrors the z-score check your `OpticaLynx` README mentioned for CLIP vs ViCLIP.

In [24]:
for name, df in results_by_backend.items():
    scores = df["score"].values
    top1 = scores[0]
    mean_rest = scores[1:].mean() if len(scores) > 1 else float("nan")
    std_rest = scores[1:].std() if len(scores) > 1 else float("nan")
    z = (top1 - mean_rest) / std_rest if std_rest > 0 else float("nan")
    print(
        f"{name:12s} top1={top1:.4f}  mean(rest)={mean_rest:.4f}  "
        f"std(rest)={std_rest:.4f}  z-score={z:.2f}"
    )

siglip2      top1=0.1931  mean(rest)=0.1810  std(rest)=0.0033  z-score=3.66
clip_vitb32  top1=0.3991  mean(rest)=0.3508  std(rest)=0.0112  z-score=4.33
viclip       top1=0.6994  mean(rest)=0.6385  std(rest)=0.0234  z-score=2.61


## 9. (Optional, later) Ground-truth eval

Not wired up yet. When you have a ground-truth query CSV (columns like `query, gt_video_id, gt_frame_id`), here's the eval loop sketch:

```python
def evaluate(search_fn, eval_df, k_values=(1, 5, 10, 20, 100)):
    hits = {k: 0 for k in k_values}
    found_ranks = []
    for _, row in eval_df.iterrows():
        res = search_fn(row['query'], k=max(k_values))
        match = res[(res.video_id == row['gt_video_id']) & (res.frame_id == row['gt_frame_id'])]
        if len(match):
            rank = int(match.iloc[0]['rank'])
            found_ranks.append(rank)
            for k in k_values:
                if rank <= k:
                    hits[k] += 1
    n = len(eval_df)
    return {
        f'hit_rate@{k}': hits[k] / n for k in k_values
    } | {
        'mean_rank_when_found': np.mean(found_ranks) if found_ranks else None,
        'found_count': len(found_ranks),
        'total_queries': n,
    }

# eval_df = pd.read_csv('/path/to/your/eval_queries.csv')
# print('siglip2:', evaluate(search_siglip2, eval_df))
# print('clip_vitb32:', evaluate(search_clip, eval_df))
# print('viclip:', evaluate(search_viclip, eval_df))
```